# 네이버 영화리뷰 감성분석

https://github.com/e9t/nsmc

In [1]:
# 데이터로드
import pandas as pd

ratings_train_df = pd.read_csv('ratings_train.txt', sep='\t')
ratings_test_df = pd.read_csv('ratings_test.txt', sep='\t')
len(ratings_train_df), len(ratings_test_df)

(150000, 50000)

In [2]:
ratings_train_df.head()

,id,document,label
0,9976970,아 더빙.. 진짜 짜증나네요 목소리,0
1,3819312,흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나,1
2,10265843,너무재밓었다그래서보는것을추천한다,0
3,9045019,교도소 이야기구먼 ..솔직히 재미는 없다..평점 조정,0
4,6483659,사이몬페그의 익살스런 연기가 돋보였던 영화!스파이더맨에서 늙어보이기만 했던 커스틴 ...,1


In [3]:
ratings_test_df.head()

,id,document,label
0,6270596,굳 ㅋ,1
1,9274899,GDNTOPCLASSINTHECLUB,0
2,8544678,뭐야 이 평점들은.... 나쁘진 않지만 10점 짜리는 더더욱 아니잖아,0
3,6825595,지루하지는 않은데 완전 막장임... 돈주고 보기에는....,0
4,6723715,3D만 아니었어도 별 다섯 개 줬을텐데.. 왜 3D로 나와서 제 심기를 불편하게 하죠??,0


In [4]:
# 결측치 확인 후 제거
print(ratings_train_df.isnull().sum())
print(ratings_test_df.isnull().sum())

ratings_train_df = ratings_train_df.dropna(how='any')
ratings_test_df = ratings_test_df.dropna(how='any')

print(ratings_train_df.isnull().sum())
print(ratings_test_df.isnull().sum())

id          0
document    5
label       0
dtype: int64
id          0
document    3
label       0
dtype: int64
id          0
document    0
label       0
dtype: int64
id          0
document    0
label       0
dtype: int64


In [5]:
# 학습 시간을 고려해서 데이터 샘플링
ratings_train_df = ratings_train_df.sample(n=50000, random_state=42)
ratings_test_df = ratings_test_df.sample(n=5000, random_state=42)
ratings_train_df.shape, ratings_test_df.shape

((50000, 3), (5000, 3))

## 데이터전처리

In [6]:
import re
from konlpy.tag import Okt

def preprocessing(sentence, okt=Okt()):
    # 개행문자 제거
    sentence = re.sub(r'\n', ' ', sentence)
    # 한글외 문자 제거
    sentence = re.sub(r'[^가-힣ㄱ-ㅎㅏ-ㅣ\s]', '', sentence)
    # 형태소분석(어간추출)
    tokens = okt.morphs(sentence, stem=True)
    # 불용어제거
    stopwords = set(['에', '은', '는', '이', '가', '그리고', '것', '들', '수', '등', '로', '을', '를', '만', '도', '아', '의', '그', '다'])
    tokens = [token for token in tokens if token not in stopwords]
    return tokens

In [7]:
sample = "진짜 이상한 영화를 봤어요. It's weird~~~ 10000원이 아까웠어요..."
preprocessing(sample)

['진짜', '이상하다', '영화', '보다', '원', '아깝다']

In [ ]:
from tqdm import tqdm

X_train = []
y_train = []
X_test = []
y_test = []

for index, row in tqdm(ratings_train_df.iterrows()):
    doc, label = row['document'], row['label']
    X_train.append(preprocessing(doc))
    y_train.append(label)

for index, row in tqdm(ratings_test_df.iterrows()):
    doc, label = row['document'], row['label']
    X_test.append(preprocessing(doc))
    y_test.append(label)


33849it [00:56, 695.93it/s]

In [ ]:
print(X_train[:3])
print(y_train[:3])
print(X_test[:3])
print(y_test[:3])

[['원본', '최고'], ['스릴', '감', '과', '훈훈하다', '있다', '영화'], ['굉장하다', '저', '평가', '되다', '영화', '중', '하나', '라고', '생각', '함']]
[1, 1, 1]
[['찌다', '여운', '과', '하다', '인생', '최고', '미드'], ['기대', '이상', '이다'], ['신나다', '없다', '애니']]
[1, 1, 0]


In [ ]:
# 정수인코딩/패딩 처리
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

vocab_size = 20000

tokenizer = Tokenizer(num_words=vocab_size, oov_token='<OOV>')

tokenizer.fit_on_texts(X_train)
X_train_encoded = tokenizer.texts_to_sequences(X_train)
X_test_encoded = tokenizer.texts_to_sequences(X_test)

In [ ]:
print(X_train_encoded[:3])
print(X_test_encoded[:3])

[[8917, 28], [521, 327, 22, 871, 7, 2], [716, 84, 454, 12, 2, 62, 78, 138, 33, 234]]
[[1152, 268, 22, 4, 156, 28, 1576], [255, 251, 6], [1644, 5, 293]]


In [ ]:
# 리뷰길이 평균/중위값 확인
import matplotlib.pyplot as plt
import numpy as np

X_train_len = [len(seq) for seq in X_train_encoded]
X_test_len = [len(seq) for seq in X_test_encoded]

mean = np.mean(X_train_len + X_test_len)
median = np.median(X_train_len + X_test_len)
print(f'평균: {mean:.2f}, 중위값: {median}')

plt.hist(X_train_len + X_test_len)
plt.show()

ImportError: DLL load failed while importing _imaging: 애플리케이션 제어 정책에서 이 파일을 차단했습니다.

In [ ]:
max_len = 15
X_train_padded = pad_sequences(X_train_encoded, maxlen=max_len)
X_test_padded = pad_sequences(X_test_encoded, maxlen=max_len)

## 텍스트 디코딩

In [ ]:
# 단어사전
# - tokenizer.index_word
# - tokenizer.word_index

def decode_sentence(encoded_sentence):
    decoded = [tokenizer.index_word.get(n, '_') for n in encoded_sentence]
    return ' '.join(decoded)

print(X_train_padded[0])
print(decode_sentence(X_train_padded[0]))

print(ratings_test_df.iloc[0, 1])
print(X_test_padded[0])
print(decode_sentence(X_test_padded[0]))

## 모델 설계

In [ ]:
import torch
import torch.nn as nn

class NSMC(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hidden_dim * 2, 1)

    def forward(self, x):
        x = self.embedding(x)
        _, (hidden, _) = self.lstm(x)
        f_h = hidden[-2]
        b_h = hidden[-1]
        hidden = torch.cat((f_h, b_h), dim=1)
        output = self.fc(hidden)
        return output

embedding_dim = 100
hidden_dim = 128
model = NSMC(vocab_size, embedding_dim, hidden_dim)
model

In [ ]:
# 파라미터 확인
total_params = 0

for name, param in model.named_parameters():
    if param.requires_grad:
        param_count = param.numel() 
        total_params += param_count
        print(f'{name:<20} {str(list(param.shape)):<30} {param_count}')

print(f'Total Trainable Parameters: {total_params}')

## 모델 학습

In [ ]:
# tensor, dataset, dataloader준비
batch_size = 64
train_size = int(len(X_train_padded) * 0.8) 
val_size = len(X_train_padded) - train_size 

X_train_padded = torch.tensor(X_train_padded, dtype=torch.long)
X_test_padded = torch.tensor(X_test_padded, dtype=torch.long)
y_train = torch.tensor(y_train, dtype=torch.float).unsqueeze(1)
y_test = torch.tensor(y_test, dtype=torch.float).unsqueeze(1)

print(X_train_padded.shape, X_test_padded.shape)
print(y_train.shape, y_test.shape)

In [ ]:
from torch.utils.data import TensorDataset, DataLoader, random_split

train_dataset = TensorDataset(X_train_padded, y_train)
train_dataset, val_dataset = random_split(train_dataset, [train_size, val_size])
test_dataset = TensorDataset(X_test_padded, y_test)

train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_dataloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(len(train_dataset), len(val_dataset), len(test_dataset))
print(len(train_dataloader), len(val_dataloader), len(test_dataloader))

In [ ]:
# 학습/검증
import torch.optim as optim

model = NSMC(vocab_size, embedding_dim, hidden_dim)

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=3
)

train_losses, val_losses, train_accs, val_accs = [], [], [], []

# 조기종료 관련
early_stopping_patience = 1
best_val_loss = float('inf')
early_stopping_counter = 0

epochs = 100

for epoch in range(epochs):
    model.train()
    total_loss, correct, total = 0, 0, 0

    for x_batch, y_batch in train_dataloader:

        optimizer.zero_grad()
        output = model(x_batch)
        loss = criterion(output, y_batch)
        loss.backward()
        optimizer.step()

        total_loss += loss.detach().item()
        p = torch.sigmoid(output)
        pred = (p >= 0.5).float()
        correct += (pred == y_batch).sum().float().detach().item()
        total += len(y_batch)

    # 학습 기록
    train_loss = total_loss / len(train_dataloader) 
    train_acc = correct / total
    train_losses.append(train_loss)
    train_accs.append(train_acc)

    # 검증
    model.eval()
    val_loss, val_correct, val_total = 0, 0, 0
    with torch.no_grad():
        for x_batch, y_batch in val_dataloader:
            output = model(x_batch)
            loss = criterion(output, y_batch)

            val_loss += loss.detach().item()
            p = torch.sigmoid(output)
            pred = (p >= 0.5).float()
            val_correct += (pred == y_batch).sum().float().detach().item()
            val_total += len(y_batch)

    val_loss = val_loss / len(val_dataloader)
    val_acc = val_correct / val_total
    val_losses.append(val_loss)
    val_accs.append(val_acc)

    # 스케쥴러 검사
    scheduler.step(val_loss)
    lr = scheduler.get_last_lr()[0]
    
    print(
        f'Epoch({epoch + 1}/{epochs}): '
        f'Train Loss {train_loss:.4f}, '
        f'Train Acc {train_acc:.4f}, '
        f'Val Loss {val_loss:.4f}, '
        f'Val Acc {val_acc:.4f}, '
        f'LR {lr}'
    )

    # 조기종료
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        early_stopping_counter = 0
    else:
        early_stopping_counter += 1
        if early_stopping_counter >= early_stopping_patience:
            print(f'Early stopped at Epoch {epoch + 1}...')
            break

In [ ]:
train_losses, val_losses, train_accs, val_accs

In [ ]:
# 시각화
import matplotlib.pyplot as plt

history_df = pd.DataFrame({
    'train_loss': train_losses,
    'train_acc': train_accs,
    'val_loss': val_losses,
    'val_acc': val_accs
})
history_df.plot()
plt.show()

## 모델 평가

In [ ]:
model.eval()
test_loss, test_correct, test_total = 0, 0, 0

with torch.no_grad():
    for x_batch, y_batch in test_dataloader:

        output = model(x_batch)
        loss = criterion(output, y_batch)
        total_loss += loss.detach().item()

        p = torch.sigmoid(output)
        pred = (p >= 0.5).float()
        correct = (pred == y_batch).sum().detach().item()
        test_correct += correct
        test_total += len(y_batch)

    test_loss = total_loss / len(test_dataloader)
    test_acc = test_correct / test_total

print(f'Testset: Loss {test_loss:.4f}, Acc {test_acc:.4f}')

## 모델 추론

In [ ]:
def movie_review_sentimental_analysis(sentences):
    
    # 토큰화/전처리
    tokens = [preprocessing(sent) for sent in sentences]
    # 정수인코딩/패딩처리
    encoded_sequences = tokenizer.texts_to_sequences(tokens)
    padded_sequences = pad_sequences(encoded_sequences, maxlen=max_len)
    # 텐서 변환
    X = torch.tensor(padded_sequences, dtype=torch.long)

    model.eval()
    with torch.no_grad():
        output = model(X)
        prob = torch.sigmoid(output)
        pred = (prob >= 0.5).float()

        return ['긍정' if pr == 1 else '부정' for pr in pred]

samples = [
    '이 영화는 정말 짜증나네요.',
    "오랜만에 수작을 만났습니다.",
    "내가 만들어도 이거보다는 ...",
    "고구마 영화",
    "황정민이 최고야!",
    "너무 유치해서 오글거려요~"
]

movie_review_sentimental_analysis(samples)